In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


%matplotlib inline

In [ ]:
# Task 1: Write your code here:
# read datset

delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)


In [ ]:
# Task 2: Write your code here:
# show first rows

print(f"Dataset shape: {df_delivery.shape}")
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
# display dataset info

df_delivery.info()

In [ ]:
# Task 4: Write your code here:
# show statistical description using describe
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
# plot the target (delivery time)

def check_target_distribution(df, target_column):
  df_delivery[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()
# plot or print distribution of target variable
check_target_distribution(df_delivery, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# drop order id column

df_delivery.columns.drop("Order_ID")


In [ ]:
# Task 2: Write your code here:
# handle missing values
def check_missing_values(df_delivery):
  missing_values = df_delivery.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery)

In [ ]:
# fill missing values with mode
for col in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']:
     df_delivery[col] = df_delivery[col].fillna(df_delivery[col].mode()[0])

# total missing values in whole dataset
print("Missing values remaining:", df_delivery.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
# check and remove duplicates

def check_duplicates(df_delivery):
  duplicates = df_delivery.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_delivery.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
# Task 4: Write your code here:
# encode categorical variables

# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_delivery[col] = le.fit_transform(df_delivery[col].astype(str))


df_delivery.head()

In [ ]:
# Task 5: Write your code here:
# apply feature scaling using standard scaler

print('data before scaling:\n', df_delivery) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df_delivery) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
# target imbalance
def check_target_imbalance(df_delivery, target_column):
  print("Target Distribution:")
  print(df_delivery[target_column].value_counts(normalize=True))
  sns.countplot(x=df_delivery[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_delivery, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# split dataset
features = ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_delivery[features]
y = df_delivery['Delivery_Time']

# split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # shuffle data before splitting -- representative splits
    stratify=y
    )

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# get min and max across all features
print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
# convert numpy array to dataframe with original columns names
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

# returns indices for train and validation sets for each fold
for train_idx, val_idx in kfold.split(X_train_scaled):
    # split train into train and validation sets
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train) # each tree learns patterns between features and price
print("Model trained!")


# Predict and evaluate
y_pred = model.predict(X_test_scaled)

# for model accuracy test, evaluate performance
mae = mean_absolute_error(y_test, y_pred) # between actual and predicted y
# Train and predict
model.fit(X_fold_train, y_fold_train)
y_fold_pred = model.predict(X_fold_val)

# Calculate metrics
mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


# for model accuracy test, evaluate performance
mae = mean_absolute_error(y_test, y_pred) # between actual and predicted y

print(f"MAE:  ${mae:,.2f}")





In [ ]:
# Task 1: Write your code here:
# Feature importance - which features contribute more to predicting target
# list of column names used to train model, importance from randon forest for each feature, sort features from most to least important
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importance_
}).sort_values('importance', ascending=False)


plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
# Ensures the most important feature appears at the top.
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: